In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [2]:
import json
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import numpy as np

In [3]:
# -------------------------------------------------------------
# 1. Count how many groups use each technique
# -------------------------------------------------------------
tech_group_counts = (
    group_techniques_df.groupby('technique_id')['group_id']
                       .nunique()
                       .reset_index(name='num_groups_using')
)

# -------------------------------------------------------------
# 2. Compute inverse frequency for each technique
# -------------------------------------------------------------
tech_group_counts['inv_freq'] = 1 / tech_group_counts['num_groups_using']

# -------------------------------------------------------------
# 3. Merge inverse frequency onto group_techniques_df
# -------------------------------------------------------------
group_tech_novelty = group_techniques_df.merge(
    tech_group_counts[['technique_id', 'inv_freq']],
    on='technique_id',
    how='left'
)

# -------------------------------------------------------------
# 4. Compute average inverse frequency per group
# -------------------------------------------------------------
novelty_df = (
    group_tech_novelty.groupby(['group_id', 'group_name'])['inv_freq']
                      .mean()
                      .reset_index(name='avg_inv_freq')
)

# -------------------------------------------------------------
# 5. Normalize the score to 0–1
# -------------------------------------------------------------
novelty_df['scaled_score'] = (
    (novelty_df['avg_inv_freq'] - novelty_df['avg_inv_freq'].min()) /
    (novelty_df['avg_inv_freq'].max() - novelty_df['avg_inv_freq'].min())
)

# -------------------------------------------------------------
# 6. Sort from most → least novel
# -------------------------------------------------------------
novelty_df = novelty_df.sort_values('scaled_score', ascending=False).reset_index(drop=True)

novelty_df


,group_id,group_name,avg_inv_freq,scaled_score
0,intrusion-set--96e239be-ad99-49eb-b127-3007b8c...,Equation,0.527778,1.000000
1,intrusion-set--277d2f87-2ae5-4730-a3aa-50c1fdf...,Strider,0.370370,0.694551
2,intrusion-set--a0cb9370-e39b-44d5-9f50-ef78e41...,Axiom,0.268560,0.496988
3,intrusion-set--d8bc9788-4f7d-41a9-9e9d-ee1ea18...,LAPSUS$,0.266856,0.493683
4,intrusion-set--461b8e25-8f4a-4ea2-a4a8-e39df7c...,UNC3886,0.253340,0.467455
...,...,...,...,...
163,intrusion-set--fed4f0a2-4347-4530-b0f5-6dfd49b...,Nomadic Octopus,0.025041,0.024441
164,intrusion-set--03506554-5f37-4f8f-9ce4-0e9f01a...,Elderwood,0.023149,0.020770
165,intrusion-set--fe98767f-9df8-42b9-83c9-004b1de...,PittyTiger,0.017693,0.010182
166,intrusion-set--62a64fd3-aaf7-4d09-a375-d6f8bb1...,TA459,0.016710,0.008275


In [8]:
novelty_df[['group_name', 'scaled_score']].head(20)

,group_name,scaled_score
0,Equation,1.000000
1,Strider,0.694551
2,Axiom,0.496988
3,LAPSUS$,0.493683
4,UNC3886,0.467455
5,APT29,0.450093
6,Storm-0501,0.429543
7,TeamTNT,0.429504
8,Contagious Interview,0.415203
9,APT12,0.407331


CSV exporting

In [4]:
# csv_df = novelty_df[['group_name', 'scaled_score']]
# csv_df.rename(columns={'scaled_score': 'score'}, inplace=True)
# csv_df.to_csv("../analysis_data/technique_novelty.csv", index=False)

In [ ]:
# # -------------------------------------------------------------
# # 1. Count how many groups use each technique
# # -------------------------------------------------------------
# tech_group_counts = (
#     group_techniques_df.groupby('technique_id')['group_id']
#                        .nunique()
#                        .reset_index(name='num_groups_using')
# )

# # -------------------------------------------------------------
# # 2. Compute inverse frequency for each technique
# # -------------------------------------------------------------
# tech_group_counts['inv_freq'] = 1 / tech_group_counts['num_groups_using']

# # -------------------------------------------------------------
# # 3. Merge inverse frequency onto group_techniques_df
# # -------------------------------------------------------------
# group_tech_novelty = group_techniques_df.merge(
#     tech_group_counts[['technique_id', 'inv_freq']],
#     on='technique_id',
#     how='left'
# )

# # -------------------------------------------------------------
# # 4. Compute average inverse frequency per group
# # -------------------------------------------------------------
# novelty_df = (
#     group_tech_novelty.groupby(['group_id', 'group_name'])['inv_freq']
#                       .mean()
#                       .reset_index(name='avg_inv_freq')
# )

# # -------------------------------------------------------------
# # 5. Normalize the score to 0–1
# # -------------------------------------------------------------
# novelty_df['scaled_score'] = (
#     (novelty_df['avg_inv_freq'] - novelty_df['avg_inv_freq'].min()) /
#     (novelty_df['avg_inv_freq'].max() - novelty_df['avg_inv_freq'].min())
# )

# # -------------------------------------------------------------
# # 6. Sort from most → least novel
# # -------------------------------------------------------------
# novelty_df = novelty_df.sort_values('scaled_score', ascending=False).reset_index(drop=True)

# tech_group_counts_named = tech_group_counts.merge(
#     tech_df[['id', 'tech_name']],
#     left_on='technique_id',
#     right_on='id',
#     how='left'
# ).drop(columns=['id'])  # drop duplicate ID column

# # -------------------------------------------------------------
# # 8. Identify high-novelty and low-novelty techniques
# # -------------------------------------------------------------
# # Top 10 most novel (used by very few groups)
# most_novel_techniques = (
#     tech_group_counts_named.sort_values('inv_freq', ascending=False)
#                            .head(10)
#                            .reset_index(drop=True)
# )

# # Bottom 10 least novel (used by many groups)
# least_novel_techniques = (
#     tech_group_counts_named.sort_values('inv_freq', ascending=True)
#                            .head(10)
#                            .reset_index(drop=True)
# )

# most_novel_techniques, least_novel_techniques, novelty_df


(                                        technique_id  num_groups_using  \
 0  attack-pattern--ffeb0780-356e-4261-b036-cfb6bd...                 1   
 1  attack-pattern--79a47ad0-fc3b-4821-9f01-a026b1...                 1   
 2  attack-pattern--6e3bd510-6b33-41a4-af80-2d80f3...                 1   
 3  attack-pattern--6e561441-8431-4773-a9b8-ccf28e...                 1   
 4  attack-pattern--7007935a-a8a7-4c0b-bd98-4e85be...                 1   
 5  attack-pattern--70d81154-b187-45f9-8ec5-295d01...                 1   
 6  attack-pattern--73b24a10-6bf4-4af1-a81e-67b8bc...                 1   
 7  attack-pattern--768dce68-8d0d-477a-b01d-0eea98...                 1   
 8  attack-pattern--774ad5bb-2366-4c13-a8a9-65e50b...                 1   
 9  attack-pattern--77e29a47-e263-4f11-8692-e5012f...                 1   
 
    inv_freq                                       tech_name  
 0       1.0                                    COR_PROFILER  
 1       1.0                          Office Te

In [ ]:
# most_novel_techniques.head(50)

,technique_id,num_groups_using,inv_freq,tech_name
0,attack-pattern--ffeb0780-356e-4261-b036-cfb6bd...,1,1.0,COR_PROFILER
1,attack-pattern--79a47ad0-fc3b-4821-9f01-a026b1...,1,1.0,Office Template Macros
2,attack-pattern--6e3bd510-6b33-41a4-af80-2d80f3...,1,1.0,Odbcconf
3,attack-pattern--6e561441-8431-4773-a9b8-ccf28e...,1,1.0,Search Engines
4,attack-pattern--7007935a-a8a7-4c0b-bd98-4e85be...,1,1.0,Process Doppelgänging
5,attack-pattern--70d81154-b187-45f9-8ec5-295d01...,1,1.0,Executable Installer File Permissions Weakness
6,attack-pattern--73b24a10-6bf4-4af1-a81e-67b8bc...,1,1.0,Malicious Library
7,attack-pattern--768dce68-8d0d-477a-b01d-0eea98...,1,1.0,Golden Ticket
8,attack-pattern--774ad5bb-2366-4c13-a8a9-65e50b...,1,1.0,Client Configurations
9,attack-pattern--77e29a47-e263-4f11-8692-e5012f...,1,1.0,IDE Tunneling


In [ ]:
# least_novel_techniques.head(50)

,technique_id,num_groups_using,inv_freq,tech_name
0,attack-pattern--e6919abc-99f9-4c6c-95a5-14761e...,85,0.011765,Ingress Tool Transfer
1,attack-pattern--232b7f21-adf9-4b42-b936-b9d6f7...,84,0.011905,Malicious File
2,attack-pattern--970a3432-3237-47ad-bcca-7d8cbb...,83,0.012048,PowerShell
3,attack-pattern--a2fdce72-04b2-409a-ac10-cc1695...,79,0.012658,Tool
4,attack-pattern--2e34237d-8574-43f6-aace-ae2915...,77,0.012987,Spearphishing Attachment
5,attack-pattern--d1fcf083-a721-4223-aedf-bf8960...,71,0.014085,Windows Command Shell
6,attack-pattern--1c4e5d32-1fe9-4116-9d9d-59e392...,59,0.016949,Match Legitimate Resource Name or Location
7,attack-pattern--df8b2a25-8bdf-4856-953c-a04372...,56,0.017857,Web Protocols
8,attack-pattern--354a7f88-63fb-41b5-a801-ce3b37...,55,0.018182,System Information Discovery
9,attack-pattern--9efb1ea7-c37b-4595-9640-b7680c...,55,0.018182,Registry Run Keys / Startup Folder
